# Group 5: ETL Pipeline

After initial discussions with the client, we learned that all operational data has historically been recorded and stored in various Excel spreadsheets. To successfully migrate this data into our newly designed relational database schema, we propose a structured, multi-step process. 
* First, we will analyze the existing spreadsheets to map the available columns to the appropriate tables and fields in the new schema.
* Second, we will perform necessary data cleaning and transformations — including splitting combined fields, formatting dates, standardizing categorical variables, and handling missing values — to ensure compatibility with the database structure. Ideally we will manipulate the data prior to loading so the final import will have a smooth transition to the new database.
* Finally, we will load the cleaned datasets through csv files, matching the information that is needed to each relation in our proposed schema.

Following our review of the client's information, we identified 4 key spreadsheets that contain historical operational data. Each spreadsheet has multiple columns, which will be mapped, cleaned and migrated to our new schema.
1) ABC_employees.xlsx
   * first_name — Employee’s first name
   * last_name — Employee’s last name
   * department — Department (e.g., Produce, Meat, Bakery)
   * salary — Yearly salary
   * hours_per_week — Contracted working hours
   * hire_date — Date of hire (some missing values)
   * location_name — Store where the employee works (to be mapped to location_id)
   * shift_date — Date of scheduled shift
   * start_time — Shift start time
   * end_time — Shift end time
   * shift_status — Scheduled, absent, or on leave
2) ABC_products_and_suppliers.xlsx
   * productname — Name of the product
   * category — Product category (e.g., Dairy, Produce, Frozen)
   * unitcost — Cost per unit (historical fluctuations noted)
   * unitprice — Retail price per unit
   * expiration_days — Expected shelf life in days
   * manufacturer_name — Manufacturer's name
   * supplier_name — Supplier’s name
   * supplier_email — Supplier’s contact email
   * supplier_phone — Supplier’s contact number
   * notes — Additional supplier notes (optional)
3) ABC_orders.xlsx
   * order_date — Date the purchase order was placed
   * expected_delivery_date — Expected delivery date
   * delivery_date — Actual delivery date (may be missing for pending orders)
   * supplier_name — Supplier for the order
   * location — Store receiving the order
   * status — Status of the order (varies, may need mapping to ('Ordered', 'Received', 'Cancelled'))
   * productname — Product ordered
   * units — Units ordered
   * unit_cost — Cost per unit at the time of order
4) ABC_sales_and_customers.xlsx
   * sale_date — Date of sale
   * location_name — Store location of the sale
   * customer_first_name — Customer first name
   * customer_last_name — Customer last name
   * age — Customer age
   * gender — Customer gender
   * email — Customer email (sometimes missing)
   * loyalty_member — Whether the customer is a loyalty member (yes/no)
   * product_name — Product sold
   * quantity — Quantity sold
   * unit_price — Price at time of sale
   * discount_applied — Discount applied during sale
   * promotion_name — Name of promotion (if any)
   * notes - Notes (including 'returns')

Based from this information, we can map the data inputs from identified columns directly to our schema:
* Employees: employees and staffing
* Products and suppliers: products, manufacturers and suppliers
* Orders: purchase_orders, deliveries and purchase_order_details
* Sales and customers: customers, sales_orders, sales_orders_details and promotions

Other tables such as inventory, transactions, and returns will not be manually populated from the client spreadsheets. Instead, they will be populated through an automated process and triggers. For example:
* purchase_order_details and sales_orders_details will be populated based on items from the order data.
* inventory will automatically be updated by triggers when deliveries are finalized or when products are sold.

### Data Extraction and Transformation

#### Employees Table
We will systematically inspect each file and make necessary transformations. The main steps for this file are:
1) Checking column names and renaming if necessary
2) Checking for missing required fields (such as first_name, last_name, location_name for NOT NULL constraint)
3) Format date types to datetime
4) Standardize categorical values (such as shift_status to be in 'scheduled','Absent','On Leave')
5) Check for relational integrity

In [22]:
import pandas as pd

# Read file and see the data first
file_path = '/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx'

# Read each sheet
employees_raw = pd.read_excel(file_path, sheet_name='Employees')
staffing_raw = pd.read_excel(file_path, sheet_name='Staffing')

# Display basic info
print("=== Employees Table ===")
print(employees_raw.info())
print(employees_raw.head())

print("\n=== Staffing Table ===")
print(staffing_raw.info())
print(staffing_raw.head())

=== Employees Table ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   first_name      10 non-null     object        
 1   last_name       10 non-null     object        
 2   department      10 non-null     object        
 3   salary          10 non-null     float64       
 4   hours_per_week  10 non-null     int64         
 5   hire_date       10 non-null     datetime64[ns]
 6   location_name   10 non-null     object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 692.0+ bytes
None
  first_name last_name department    salary  hours_per_week  hire_date  \
0     Alyssa      Lane    Cashier  33065.26              20 2024-10-02   
1       Todd  Williams    Cashier  33298.05              30 2025-01-09   
2      Sandy    Howell    Grocery  33964.15              20 2024-08-19   
3      James  Robins

In [24]:
# Step 1: Rename columns if needed (to match database schema)
employees_raw.columns = employees_raw.columns.str.strip().str.lower().str.replace(' ', '_')
staffing_raw.columns = staffing_raw.columns.str.strip().str.lower().str.replace(' ', '_')

# Step 2: Check for missing important values
print("Missing values in Employees:")
print(employees_raw[['first_name', 'last_name', 'location_name']].isnull().sum())

print("\nMissing values in Staffing:")
print(staffing_raw[['first_name', 'last_name', 'shift_date', 'start_time', 'end_time']].isnull().sum())

# Optional: Drop rows with missing critical values
employees_raw.dropna(subset=['first_name', 'last_name', 'location_name'], inplace=True)
staffing_raw.dropna(subset=['first_name', 'last_name', 'shift_date', 'start_time', 'end_time'], inplace=True)

# Step 3: Format datetime columns
employees_raw['hire_date'] = pd.to_datetime(employees_raw['hire_date'], errors='coerce')
staffing_raw['shift_date'] = pd.to_datetime(staffing_raw['shift_date'], errors='coerce')

# Step 4: Standardize categorical fields
staffing_raw['shift_status'] = staffing_raw['shift_status'].str.strip().str.capitalize()
allowed_statuses = ['Scheduled', 'Absent', 'On Leave']
staffing_raw = staffing_raw[staffing_raw['shift_status'].isin(allowed_statuses)]

# Final preview
print("\n=== Cleaned Employees Data ===")
print(employees_raw.head())

print("\n=== Cleaned Staffing Data ===")
print(staffing_raw.head())

Missing values in Employees:
first_name       0
last_name        0
location_name    0
dtype: int64

Missing values in Staffing:
first_name    0
last_name     0
shift_date    0
start_time    0
end_time      0
dtype: int64

=== Cleaned Employees Data ===
  first_name last_name department    salary  hours_per_week  hire_date  \
0     Alyssa      Lane    Cashier  33065.26              20 2024-10-02   
1       Todd  Williams    Cashier  33298.05              30 2025-01-09   
2      Sandy    Howell    Grocery  33964.15              20 2024-08-19   
3      James  Robinson       Deli  34153.37              20 2024-05-23   
4      Riley    Carter    Seafood  34398.13              35 2024-05-10   

     location_name  
0  ABC Queens East  
1  ABC Queens West  
2  ABC Queens West  
3  ABC Queens West  
4  ABC Queens West  

=== Cleaned Staffing Data ===
  first_name last_name shift_date start_time end_time shift_status  \
0     Alyssa      Lane 2025-04-30      14:00    21:00       Absent   
1    

In [28]:
# Step 5: Checking relational integrity. We have to make sure that location_name matches store names in locations.
# Since there is no locations, we will create a reference table and map them.

locations_reference = pd.DataFrame({
    'location_id': [1, 2],
    'location_name': ['ABC Queens East', 'ABC Queens West']
})

print(locations_reference)

# Merge locations into employees
employees_clean = employees_raw.merge(locations_reference, how='left', on='location_name')

# Merge locations into staffing
staffing_clean = staffing_raw.merge(locations_reference, how='left', on='location_name')

# Check if any locations failed to match
print("\nMissing locations in employees after merge:", employees_clean['location_id'].isnull().sum())
print("Missing locations in staffing after merge:", staffing_clean['location_id'].isnull().sum())

# Final cleaned datasets
print("\n=== Employees with location_id ===")
print(employees_clean[['first_name', 'last_name', 'location_name', 'location_id']].head())

print("\n=== Staffing with location_id ===")
print(staffing_clean[['first_name', 'last_name', 'location_name', 'location_id']].head())

   location_id    location_name
0            1  ABC Queens East
1            2  ABC Queens West

Missing locations in employees after merge: 0
Missing locations in staffing after merge: 0

=== Employees with location_id ===
  first_name last_name    location_name  location_id
0     Alyssa      Lane  ABC Queens East            1
1       Todd  Williams  ABC Queens West            2
2      Sandy    Howell  ABC Queens West            2
3      James  Robinson  ABC Queens West            2
4      Riley    Carter  ABC Queens West            2

=== Staffing with location_id ===
  first_name last_name    location_name  location_id
0     Alyssa      Lane  ABC Queens East            1
1     Alyssa      Lane  ABC Queens East            1
2     Alyssa      Lane  ABC Queens East            1
3     Alyssa      Lane  ABC Queens East            1
4     Alyssa      Lane  ABC Queens East            1


#### Products and suppliers table
In this table, we first have to map what information goes to which relation.

Mapping: ABC_products_and_suppliers.xlsx → Target Schema

| Spreadsheet Column  | Target Table   | Target Field        |
|:--------------------|:---------------|:--------------------|
| product_name         | products        | product_name         |
| category             | products        | category             |
| unit_cost            | products        | unit_cost            |
| unit_price           | products        | unit_price           |
| expiration_days      | products        | expiration_days      |
| manufacturer_name    | manufacturers   | manufacturer_name    |
| supplier_name        | suppliers       | supplier_name        |
| supplier_email       | suppliers       | email                |
| supplier_phone       | suppliers       | phone_number         |
| notes                | suppliers       | notes                |

Once mapped, the key tasks are as follows:
1) Ensuring no missing product names (No NULL values)
2) Removing any duplciates
3) Handling missing supplier or manufacturer emails or notes (Fill with NULL for missing values)
4) Normalize pricing (Ensure that the cost is higher than the price for sanity)
5) Assign IDs (for manufacturer_id, supplier_id) and link them into products

In [30]:
import pandas as pd

# Load the raw products and suppliers spreadsheet
products_suppliers_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.xlsx')

# View basic info
print(products_suppliers_clean.info())  # Check datatypes and missing values
print(products_suppliers_clean.head())  # View sample rows

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   productname        500 non-null    object 
 1   category           500 non-null    object 
 2   unitcost           500 non-null    float64
 3   unitprice          500 non-null    float64
 4   expiration_days    500 non-null    int64  
 5   manufacturer_name  500 non-null    object 
 6   supplier_name      500 non-null    object 
 7   supplier_email     500 non-null    object 
 8   supplier_phone     500 non-null    object 
 9   notes              51 non-null     object 
dtypes: float64(2), int64(1), object(7)
memory usage: 39.2+ KB
None
                productname    category  unitcost  unitprice  expiration_days  \
0  Orange Juice (with Pulp)   Beverages       2.6       6.27               45   
1        Cream Cheese (8oz)       Dairy       1.2       2.50               60   
2    Pep

In [32]:
# Cleaning/standardizing

# Handle missing values (example: if unit_cost or unit_price is missing, flag it)
products_suppliers_clean['unitcost'] = products_suppliers_clean['unitcost'].fillna(0)
products_suppliers_clean['unitprice'] = products_suppliers_clean['unitprice'].fillna(0)
products_suppliers_clean['expiration_days'] = products_suppliers_clean['expiration_days'].fillna(30)  # Assume 30 days if missing

# Drop fully empty supplier fields if needed
products_suppliers_clean = products_suppliers_clean.dropna(subset=['supplier_name'])

# Review cleaned data
print(products_suppliers_clean.info())
print(products_suppliers_clean.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   productname        500 non-null    object 
 1   category           500 non-null    object 
 2   unitcost           500 non-null    float64
 3   unitprice          500 non-null    float64
 4   expiration_days    500 non-null    int64  
 5   manufacturer_name  500 non-null    object 
 6   supplier_name      500 non-null    object 
 7   supplier_email     500 non-null    object 
 8   supplier_phone     500 non-null    object 
 9   notes              51 non-null     object 
dtypes: float64(2), int64(1), object(7)
memory usage: 39.2+ KB
None
                productname    category  unitcost  unitprice  expiration_days  \
0  Orange Juice (with Pulp)   Beverages       2.6       6.27               45   
1        Cream Cheese (8oz)       Dairy       1.2       2.50               60   
2    Pep

#### Orders table
Again, we will first map the table to the target schema.

Mapping for ABC_orders.xlsx -> Target Schema

| Spreadsheet Column         | Target Table(s)                  | Target Field                      |
|-----------------------------|-----------------------------------|------------------------------------|
| order_date                  | purchase_orders                   | order_date                        |
| expected_delivery_date      | deliveries                        | expected_delivery_date            |
| delivery_date               | deliveries                        | delivery_date                     |
| supplier_name               | purchase_orders & deliveries      | supplier_id (after lookup)        |
| location_name               | purchase_orders                   | location_id (after lookup)        |
| status                      | purchase_orders                   | status                            |
| product_name                | purchase_order_details            | product_id (after lookup)         |
| quantity                    | purchase_order_details            | quantity                          |
| unit_cost                   | purchase_order_details            | unit_cost                         |

Once mapped, the key tasks are as follows:
1) Mapping client status to schema (pending, received -> ordered, received, cancelled)
2) Fix inconsistent supplier and location names
3) Convert dates into proper datetime format
4) Product mapping between product name and product_id from products table

In [34]:
# Loading and displaying info
import pandas as pd

orders_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx')

print(orders_clean.info())
print(orders_clean.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   order_date              1000 non-null   datetime64[ns]
 1   expected_delivery_date  1000 non-null   datetime64[ns]
 2   delivery_date           834 non-null    datetime64[ns]
 3   supplier_name           1000 non-null   object        
 4   location                1000 non-null   object        
 5   status                  1000 non-null   object        
 6   productname             1000 non-null   object        
 7   units                   1000 non-null   int64         
 8   unit_cost               1000 non-null   float64       
dtypes: datetime64[ns](3), float64(1), int64(1), object(4)
memory usage: 70.4+ KB
None
  order_date expected_delivery_date delivery_date       supplier_name  \
0 2024-03-25             2024-04-01    2024-04-03        Anderson PLC   
1 

In [38]:
# Map statuses to correct schema values
status_mapping = {
    'pending': 'Ordered',
    'shipped': 'Ordered',
    'completed': 'Received',
    'received': 'Received',
    'cancelled': 'Cancelled',
    'canceled': 'Cancelled',
    'delivered': 'Received'
}
orders_clean['status'] = orders_clean['status'].str.lower().map(status_mapping).fillna('Ordered')

# Convert date fields
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
orders_clean['expected_delivery_date'] = pd.to_datetime(orders_clean['expected_delivery_date'], errors='coerce')
orders_clean['delivery_date'] = pd.to_datetime(orders_clean['delivery_date'], errors='coerce')

# Fill missing quantities and costs
orders_clean['units'] = orders_clean['units'].fillna(1)
orders_clean['unit_cost'] = orders_clean['unit_cost'].fillna(0)

# Create clean DataFrames

# Purchase Orders (deduplicated at order level)
purchase_orders_df = orders_clean[['order_date', 'supplier_name', 'location', 'status']].drop_duplicates()

# Deliveries (deduplicated at order level)
deliveries_df = orders_clean[['order_date', 'expected_delivery_date', 'delivery_date', 'supplier_name']].drop_duplicates()

# Purchase Order Details (full product-level granularity)
purchase_order_details_df = orders_clean[['productname', 'units', 'unit_cost']]

# Check cleaned tables
print(purchase_orders_df.head())
print(deliveries_df.head())
print(purchase_order_details_df.head())

  order_date       supplier_name         location    status
0 2024-03-25        Anderson PLC  ABC Queens East  Received
1 2024-10-08           Leon-Reed  ABC Queens East  Received
2 2025-03-05  Williamson-Padilla  ABC Queens West  Received
3 2025-02-17   Jenkins-Gutierrez  ABC Queens West  Received
4 2024-10-21            Rowe Inc  ABC Queens East  Received
  order_date expected_delivery_date delivery_date       supplier_name
0 2024-03-25             2024-04-01    2024-04-03        Anderson PLC
1 2024-10-08             2024-10-11    2024-10-12           Leon-Reed
2 2025-03-05             2025-03-15    2025-03-17  Williamson-Padilla
3 2025-02-17             2025-02-27    2025-02-27   Jenkins-Gutierrez
4 2024-10-21             2024-10-29    2024-10-30            Rowe Inc
              productname  units  unit_cost
0       Canned Ham (12oz)     13       2.80
1      Basmati Rice (2lb)     51       2.00
2           Pasta (Penne)     28       0.85
3  Energy Drink (Regular)     12       1.50


#### Sales and customers table

Mapping for ABC_sales_and_customers.xlsx

| Spreadsheet Column     | Target Table(s)         | Target Field                        |
|-------------------------|-------------------------|-------------------------------------|
| sale_date               | sales_orders             | order_date                          |
| location_name           | sales_orders             | location_id (after lookup)          |
| customer_first_name     | customers                | first_name                          |
| customer_last_name      | customers                | last_name                           |
| age                     | customers                | age                                 |
| gender                  | customers                | gender                              |
| email                   | customers                | email                               |
| loyalty_member          | customers                | loyalty_member (map 'yes'/'no' to boolean) |
| product_name            | sales_orders_details       | product_id (after lookup)           |
| quantity                | sales_orders_details       | quantity                            |
| unit_price              | sales_orders_details       | unit_price                          |
| discount_applied        | sales_orders_details       | discount                            |
| promotion_name          | promotions (after mapping) | promotion_id (if available)        |

Key tasks:
1) Change loyalty member from yes/no to true/false
2) Handle missing values for promotions
3) Create sales order id (unique combination of sale_date, location, customer)

In [40]:
import pandas as pd

# Load the raw sales data
sales_customers = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')

# Preview the dataset
print(sales_customers.info())
print(sales_customers.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5127 entries, 0 to 5126
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   sale_date            5127 non-null   datetime64[ns]
 1   location_name        5127 non-null   object        
 2   customer_first_name  3636 non-null   object        
 3   customer_last_name   3636 non-null   object        
 4   age                  3636 non-null   float64       
 5   gender               3636 non-null   object        
 6   email                3636 non-null   object        
 7   loyalty_member       3636 non-null   object        
 8   product_name         5127 non-null   object        
 9   quantity             5127 non-null   int64         
 10  unit_price           5127 non-null   float64       
 11  discount_applied     5127 non-null   int64         
 12  promotion_name       0 non-null      float64       
 13  notes                51 non-null 

In [42]:
# Step 1: Standardize loyalty_member column
sales_customers['loyalty_member'] = sales_customers['loyalty_member'].map({
    'yes': True,
    'no': False
})

# Step 2: Fill missing discount_applied with 0
sales_customers['discount_applied'] = sales_customers['discount_applied'].fillna(0)

# Step 3: Create unique sales_orders_id
sales_customers['sales_orders_id'] = range(1, len(sales_customers) + 1)

print("\nCleaned dataset:")
print(sales_customers.head())


Cleaned dataset:
   sale_date    location_name customer_first_name customer_last_name   age  \
0 2024-03-11  ABC Queens East              Taylor             Harris  67.0   
1 2024-03-27  ABC Queens East           Christina             Harris  66.0   
2 2024-04-05  ABC Queens East              Robert               Hill  79.0   
3 2024-04-25  ABC Queens East             Matthew             Riddle  55.0   
4 2024-05-23  ABC Queens East                 NaN                NaN   NaN   

  gender                         email loyalty_member  \
0      O  howardmontgomery@example.com          False   
1      M     townsenddiana@example.org          False   
2      M           zdeleon@example.net          False   
3      M   simmonskimberly@example.org          False   
4    NaN                           NaN            NaN   

              product_name  quantity  unit_price  discount_applied  \
0  All-Purpose Flour (5lb)        10        3.19                 0   
1  All-Purpose Flour (5lb)    

### Data Loading

We will now load the migrate the cleaned datasets into our schema.
1) Insert current data considering dependencies in order
    * Insert data into foundation tables (no dependencies)
    * Insert data into product & supply chains
    * Insert data into sales and promotions
    * Insert data into financials and returns tables
    * Check if triggers run correctly

We then loaded all the data into each table.

In [153]:
# Breaking down the 'Employees Table' into 2 files (one for each sheet)
# Load Excel files
employees_xls = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx', sheet_name=None)

# Split each sheet into Excel
for sheet_name, df in employees_xls.items():
    df.to_excel(f'/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees_{sheet_name}.xlsx', index=False)

print("Saved Employees and Staffing sheets as separate files.")

Saved Employees and Staffing sheets as separate files.


In [46]:
# 1) Locations
import psycopg2

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

employees_raw = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees_Employees.xlsx')

# Extract unique location names
unique_locations = employees_raw['location_name'].dropna().unique()

# Insert unique locations
for loc in unique_locations:
    cur.execute("""
        INSERT INTO locations (location_name, address, city, state, zip_code, opened_date, type)
        VALUES (%s, NULL, NULL, NULL, NULL, NULL, 'Other')
        ON CONFLICT (location_name) DO NOTHING;
    """, (loc,))
conn.commit()

print("Unique locations inserted successfully.")

# 5. Close cursor (optional now)
cur.close()
conn.close()

Unique locations inserted successfully.


In [164]:
# Since we only retrieved unique names and indexed them, we will manually update missing info
# We will also include the expected 3 stores in Brooklyn

import sys
from dateutil.relativedelta import relativedelta

# --- Database Connection Details ---
db_config = {
    'dbname': 'sql_final_project',
    'user': 'postgres',
    'password': '123', 
    'host': 'localhost',
    'port': '5432'
}

# --- Data for Updates (Existing Queens Stores) ---
updates = [
    {
        'location_name': 'ABC Queens East',
        'address': '70-10 Main St', 
        'city': 'Flushing', 
        'state': 'NY',
        'zip_code': '11367', 
        'opened_date': date(2024, 3, 1), 
        'type': 'Queens' 
    },
    {
        'location_name': 'ABC Queens West',
        'address': '30-05 Astoria Blvd',
        'city': 'Astoria', 
        'state': 'NY',
        'zip_code': '11102',
        'opened_date': date(2024, 6, 1), 
        'type': 'Queens' 
    }
]

# --- Data for Inserts (New Brooklyn Stores) ---
# Calculate future opening date (6 months from today)
today = date.today()
future_opened_date = today + relativedelta(months=+6)

# *** CHANGED type to 'Brooklyn' ***
inserts = [
    {
        'location_name': 'ABC Brooklyn South',
        'address': '123 Smith St', 
        'city': 'Brooklyn', 
        'state': 'NY',
        'zip_code': '11201', 
        'opened_date': future_opened_date,
        'type': 'Brooklyn'
    },
    {
        'location_name': 'ABC Brooklyn North',
        'address': '456 Bedford Ave', 
        'city': 'Brooklyn', 
        'state': 'NY',
        'zip_code': '11211', 
        'opened_date': future_opened_date,
        'type': 'Brooklyn' 
    },
    {
        'location_name': 'ABC Brooklyn West',
        'address': '789 Court St', 
        'city': 'Brooklyn', 
        'state': 'NY',
        'zip_code': '11231', 
        'opened_date': future_opened_date,
        'type': 'Brooklyn' 
    }
]


# --- Database Operations ---
conn = None
cur = None

try:
    # Connect to the database
    print(f"Connecting to database '{db_config['dbname']}'...")
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    print("Connection successful.")

    # Start transaction
    print("\nStarting transaction...")

    # 1. Perform Updates
    print("Updating existing Queens locations...")
    update_sql = """
        UPDATE locations
        SET address = %s,
            city = %s,
            state = %s,
            zip_code = %s,
            opened_date = %s,
            type = %s
        WHERE location_name = %s;
    """
    for loc in updates:
        try:
            print(f"  - Attempting update for '{loc['location_name']}' with type '{loc['type']}'") # Debug print
            cur.execute(update_sql, (
                loc['address'],
                loc['city'],
                loc['state'],
                loc['zip_code'],
                loc['opened_date'],
                loc['type'],
                loc['location_name']
            ))
            if cur.rowcount == 0:
                print(f"  - Warning: Location '{loc['location_name']}' not found for update.")
            else:
                print(f"  - Updated '{loc['location_name']}'.")
        except psycopg2.Error as e:
            print(f"  - Error updating '{loc['location_name']}': {e}")
            print(f"  - Offending type value: {loc['type']}") # Debug print
            raise # Re-raise the error to trigger rollback

    # 2. Perform Inserts
    print("\nInserting new Brooklyn locations...")
    insert_sql = """
        INSERT INTO locations (location_name, address, city, state, zip_code, opened_date, type)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (location_name) DO NOTHING; -- Avoid error if it somehow exists
    """
    for loc in inserts:
         try:
            print(f"  - Attempting insert for '{loc['location_name']}' with type '{loc['type']}'") # Debug print
            cur.execute(insert_sql, (
                loc['location_name'],
                loc['address'],
                loc['city'],
                loc['state'],
                loc['zip_code'],
                loc['opened_date'],
                loc['type']
            ))
            # Check if insert happened or conflict occurred
            if cur.rowcount == 0:
                 print(f"  - Skipped inserting '{loc['location_name']}' (already exists or conflict).")
            else:
                 print(f"  - Inserted '{loc['location_name']}' with planned opening date {loc['opened_date']}.")
         except psycopg2.Error as e:
            print(f"  - Error inserting '{loc['location_name']}': {e}")
            print(f"  - Offending type value: {loc['type']}") # Debug print
            raise # Re-raise the error to trigger rollback

    # Commit transaction if all operations were successful
    print("\nCommitting transaction...")
    conn.commit()
    print("Transaction committed successfully.")

except psycopg2.OperationalError as e:
    print(f"\nDatabase Connection Error: {e}")
    print("Please check database server status and connection details.")
except psycopg2.Error as e:
    print(f"\nDatabase Error during transaction: {e}")
    print("Rolling back transaction.")
    if conn:
        conn.rollback()
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")
    if conn:
        conn.rollback()
finally:
    # Ensure the cursor and connection are closed
    if cur:
        cur.close()
    if conn:
        conn.close()

Connecting to database 'sql_final_project'...
Connection successful.

Starting transaction...
Updating existing Queens locations...
  - Attempting update for 'ABC Queens East' with type 'Queens'
  - Updated 'ABC Queens East'.
  - Attempting update for 'ABC Queens West' with type 'Queens'
  - Updated 'ABC Queens West'.

Inserting new Brooklyn locations...
  - Attempting insert for 'ABC Brooklyn South' with type 'Brooklyn'
  - Skipped inserting 'ABC Brooklyn South' (already exists or conflict).
  - Attempting insert for 'ABC Brooklyn North' with type 'Brooklyn'
  - Skipped inserting 'ABC Brooklyn North' (already exists or conflict).
  - Attempting insert for 'ABC Brooklyn West' with type 'Brooklyn'
  - Skipped inserting 'ABC Brooklyn West' (already exists or conflict).

Committing transaction...
Transaction committed successfully.


In [50]:
# 2) Loading suppliers

df = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.xlsx')
unique_suppliers = (
    df[['supplier_name','supplier_email','supplier_phone','notes']]
    .drop_duplicates(subset=['supplier_email'])
)

conn = psycopg2.connect(dbname="sql_final_project", user="postgres",
                        password="123", host="localhost")
cur = conn.cursor()

# Insert
for _, row in unique_suppliers.iterrows():
    supplier_name = row['supplier_name'][:100] if pd.notna(row['supplier_name']) else None
    email         = row['supplier_email'][:100] if pd.notna(row['supplier_email']) else None
    phone_number  = row['supplier_phone'][:20] if pd.notna(row['supplier_phone']) else None
    notes         = row['notes'] if pd.notna(row['notes']) else None

    cur.execute("""
        INSERT INTO suppliers (supplier_name, email, phone_number, notes)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (email) DO NOTHING;
    """, (supplier_name, email, phone_number, notes))

conn.commit()
cur.close()
conn.close()
print("Suppliers inserted successfully (duplicates skipped).")

Suppliers inserted successfully (duplicates skipped).


In [52]:
# 3) Loading manufacturers

conn = psycopg2.connect(
    dbname="sql_final_project",
    user="postgres",
    password="123",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# Load manufacturers data from the products_and_suppliers
products_suppliers_df = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.xlsx')

# Extract and deduplicate manufacturer names
unique_manufacturers = products_suppliers_df[['manufacturer_name']].drop_duplicates()

# Insert each manufacturer into the 'manufacturers' table
for idx, row in unique_manufacturers.iterrows():
    cur.execute("""
        INSERT INTO manufacturers (manufacturer_name)
        VALUES (%s)
    """, (row['manufacturer_name'],))

# Commit and close
conn.commit()
cur.close()
conn.close()

print("Manufacturers inserted successfully.")

Manufacturers inserted successfully.


In [54]:
# 4) Loading Products
# A 'unique' product is classified as a unique name, category, cost, price, manufacturer_id

import pandas as pd
import psycopg2

# 1) Read the full products & suppliers sheet
file_path = '/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_products_and_suppliers.xlsx'
df = pd.read_excel(file_path)

# 2) De-duplicate: keep one row per unique product definition
unique_products = df.drop_duplicates(subset=[
    'productname',
    'category',
    'unitcost',
    'unitprice',
    'expiration_days',
    'manufacturer_name',
    'supplier_name',
])

print(f"Found {len(unique_products)} unique products;")

# 3) Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="sql_final_project",
    user="postgres",
    password="123",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# 4) Loop over unique products and insert
for _, row in unique_products.iterrows():
    # Lookup manufacturer_id
    cur.execute(
        "SELECT manufacturer_id FROM manufacturers WHERE manufacturer_name = %s;",
        (row['manufacturer_name'],)
    )
    manufacturer = cur.fetchone()
    
    # Lookup supplier_id
    cur.execute(
        "SELECT supplier_id FROM suppliers WHERE supplier_name = %s;",
        (row['supplier_name'],)
    )
    supplier = cur.fetchone()

    # Skip if lookup failed
    if manufacturer is None:
        print(f" Manufacturer not found for '{row['manufacturer_name']}' – skipping")
        continue
    if supplier is None:
        print(f" Supplier not found for '{row['supplier_name']}' – skipping")
        continue

    manufacturer_id = manufacturer[0]
    supplier_id     = supplier[0]

    # Insert into products
    cur.execute("""
        INSERT INTO products
          (product_name, category, unit_cost, unit_price, expiration_days, manufacturer_id, supplier_id)
        VALUES (%s, %s, %s, %s, %s, %s, %s);
    """, (
        row['productname'],
        row['category'],
        row['unitcost'],
        row['unitprice'],
        row['expiration_days'],
        manufacturer_id,
        supplier_id
    ))

# 5) Finalize
conn.commit()
cur.close()
conn.close()

print("Products inserted successfully!")  

Found 44 unique products;
Products inserted successfully!


In [56]:
# 5) Loading purchase orders

orders_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in orders_clean.iterrows():
    # Lookup supplier_id
    cur.execute("SELECT supplier_id FROM suppliers WHERE supplier_name = %s", (row['supplier_name'],))
    supplier_result = cur.fetchone()
    if supplier_result is None:
        print(f"Warning: Supplier '{row['supplier_name']}' not found, skipping this row.")
        continue
    supplier_id = supplier_result[0]

    # Lookup location_id
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found, skipping this row.")
        continue
    location_id = location_result[0]

    # Prepare delivery_date safely
    delivery_date = None if pd.isna(row['delivery_date']) else row['delivery_date']

    # Insert into purchase_orders
    try:
        cur.execute("""
            INSERT INTO purchase_orders (supplier_id, location_id, order_date, delivery_date, status)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            supplier_id,
            location_id,
            row['order_date'],
            delivery_date,   # This will be None if NaN
            row['status']
        ))
    except Exception as e:
        print(f"Error inserting row {index}: {e}")

conn.commit()
cur.close()
conn.close()

print("Purchase orders inserted successfully!")

Purchase orders inserted successfully!


In [58]:
# 6) Deliveries

orders_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Insert each delivery
for index, row in orders_clean.iterrows():
    # Lookup supplier_id
    cur.execute("SELECT supplier_id FROM suppliers WHERE supplier_name = %s", (row['supplier_name'],))
    supplier_result = cur.fetchone()
    if supplier_result is None:
        print(f"Warning: Supplier '{row['supplier_name']}' not found, skipping this row.")
        continue
    supplier_id = supplier_result[0]

    # Lookup purchase_order_id
    cur.execute("""
        SELECT purchase_order_id FROM purchase_orders 
        WHERE supplier_id = %s AND order_date = %s
    """, (supplier_id, row['order_date']))
    purchase_order_result = cur.fetchone()
    if purchase_order_result is None:
        print(f"Warning: Purchase order not found for supplier '{row['supplier_name']}' on {row['order_date']}, skipping.")
        continue
    purchase_order_id = purchase_order_result[0]

    # Insert into deliveries
    try:
        cur.execute("""
            INSERT INTO deliveries (purchase_order_id, supplier_id, expected_delivery_date, delivery_date, status)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            purchase_order_id,
            supplier_id,
            row['expected_delivery_date'],
            None if pd.isna(row['delivery_date']) else row['delivery_date'],
            'Scheduled' if row['status'] == 'Ordered' else 
            'Delivered' if row['status'] == 'Received' else
            'Delayed'
        ))
    except Exception as e:
        print(f"Error inserting delivery row {index}: {e}")

conn.commit()
cur.close()
conn.close()

print("Deliveries inserted successfully!")

Deliveries inserted successfully!


In [60]:
# 7) Loading purchase order details

orders_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in orders_clean.iterrows():
    # Find purchase_order_id using JOINs
    cur.execute("""
        SELECT po.purchase_order_id
        FROM purchase_orders po
        JOIN suppliers s ON po.supplier_id = s.supplier_id
        JOIN locations l ON po.location_id = l.location_id
        WHERE s.supplier_name = %s
          AND l.location_name = %s
          AND po.order_date = %s
        LIMIT 1;
    """, (row['supplier_name'], row['location'], row['order_date']))
    purchase_order_result = cur.fetchone()
    
    if purchase_order_result is None:
        print(f"Purchase order not found for row {index}. Skipping.")
        continue
    purchase_order_id = purchase_order_result[0]

    # Lookup product_id
    cur.execute("""
        SELECT product_id
        FROM products
        WHERE product_name = %s
        LIMIT 1;
    """, (row['productname'],))
    product_result = cur.fetchone()
    if product_result is None:
        print(f"Product '{row['productname']}' not found for row {index}. Skipping.")
        continue
    product_id = product_result[0]

    # Insert into purchase_order_details
    try:
        cur.execute("""
            INSERT INTO purchase_order_details (purchase_order_id, product_id, quantity, unit_cost)
            VALUES (%s, %s, %s, %s);
        """, (
            purchase_order_id,
            product_id,
            row['units'],
            row['unit_cost']
        ))
    except Exception as e:
        print(f"Error inserting row {index}: {e}")

conn.commit()
cur.close()
conn.close()

print("Purchase order details inserted successfully!")

Purchase order details inserted successfully!


In [62]:
# 8) Loading inventory

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Load the orders CSV
purchase_orders = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx')

for index, row in purchase_orders.iterrows():
    if row['status'] != 'Received':
        continue

    # Find supplier_id
    cur.execute("SELECT supplier_id FROM suppliers WHERE supplier_name = %s LIMIT 1", (row['supplier_name'],))
    supplier_result = cur.fetchone()
    if supplier_result is None:
        print(f"Warning: Supplier '{row['supplier_name']}' not found.")
        continue
    supplier_id = supplier_result[0]

    # Find location_id
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s LIMIT 1", (row['location'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found.")
        continue
    location_id = location_result[0]

    # Find product_id
    cur.execute("SELECT product_id FROM products WHERE product_name = %s LIMIT 1", (row['productname'],))
    product_result = cur.fetchone()
    if product_result is None:
        print(f"Warning: Product '{row['product_name']}' not found.")
        continue
    product_id = product_result[0]

    # Step 4: Find purchase_order_id
    cur.execute("""
        SELECT purchase_order_id
        FROM purchase_orders
        WHERE supplier_id = %s
        AND location_id = %s
        AND order_date = %s
        LIMIT 1
    """, (supplier_id, location_id, row['order_date']))
    purchase_order_result = cur.fetchone()
    if purchase_order_result is None:
        print(f"Warning: Purchase Order not found for supplier '{row['supplier_name']}' at '{row['location']}' on '{row['order_date']}'.")
        continue
    purchase_order_id = purchase_order_result[0]

    # Find delivery_id
    cur.execute("""
        SELECT delivery_id
        FROM deliveries
        WHERE purchase_order_id = %s
        LIMIT 1
    """, (purchase_order_id,))
    delivery_result = cur.fetchone()
    if delivery_result is None:
        print(f"Warning: Delivery not found for PO ID {purchase_order_id}.")
        continue
    delivery_id = delivery_result[0]

    # Calculate expiration date
    cur.execute("SELECT expiration_days FROM products WHERE product_id = %s", (product_id,))
    expiration_days_result = cur.fetchone()
    expiration_days = expiration_days_result[0] if expiration_days_result else None

    if expiration_days:
        cur.execute("SELECT delivery_date FROM deliveries WHERE delivery_id = %s", (delivery_id,))
        delivery_date_result = cur.fetchone()
        if delivery_date_result and delivery_date_result[0]:
            delivery_date = delivery_date_result[0]
            expiration_date = delivery_date + pd.Timedelta(days=expiration_days)
        else:
            expiration_date = None
    else:
        expiration_date = None

    # Insert into inventory
    try:
        cur.execute("""
            INSERT INTO inventory (product_id, location_id, quantity, entry_date, expiration_date, purchase_order_id, delivery_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            product_id,
            location_id,
            row['units'],
            row['delivery_date'],
            expiration_date,
            purchase_order_id,
            delivery_id
        ))
    except Exception as e:
        print(f"Error inserting inventory row {index}: {e}")
        conn.rollback()
        continue

conn.commit()
cur.close()
conn.close()

print("Inventory inserted successfully!")

Inventory inserted successfully!


Some values were skipped for purchase orders as the delivery date might still not be delivered.

In [155]:
# 9) Employees

# --- Database Connection Details ---
db_config = {
    'dbname': 'sql_final_project',
    'user': 'postgres',
    'password': '123', 
    'host': 'localhost',
    'port': '5432'
}

# --- File Path ---
file_path = '/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx'
sheet_name_to_read = 'Employees' 

# --- Load Data ---
try:
    employees_clean = pd.read_excel(file_path, sheet_name=sheet_name_to_read)
    print(f"Successfully read {len(employees_clean)} rows from sheet '{sheet_name_to_read}' in {file_path}")
    # *** Check if the source 'salary' column exists in Excel ***
    if 'salary' not in employees_clean.columns:
        print(f"Error: Column 'salary' not found in {file_path}, sheet '{sheet_name_to_read}'. Found columns: {employees_clean.columns.tolist()}")
        print("Please ensure the Excel file was generated with the script that includes 'salary'.")
        sys.exit()
    if 'hire_date' in employees_clean.columns:
        employees_clean['hire_date'] = pd.to_datetime(employees_clean['hire_date'], errors='coerce')
        print("Converted 'hire_date' column, handling potential errors.")
    else:
         print("Warning: 'hire_date' column not found.")

except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    sys.exit()
except ValueError as e:
     print(f"Error: Sheet '{sheet_name_to_read}' not found in {file_path}. Details: {e}")
     sys.exit()
except Exception as e:
    print(f"Error reading Excel file: {e}")
    sys.exit()


# --- Database Operations ---
conn = None
cur = None
success = False

try:
    print(f"Connecting to database '{db_config['dbname']}'...")
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    print("Connection successful.")

    # *** UPDATED SQL INSERT statement to target 'yearly_salary' column in DB ***
    # Make sure your 'employees' table has a 'yearly_salary' column (e.g., NUMERIC(10, 2))
    insert_sql = """
        INSERT INTO employees (location_id, first_name, last_name, department, yearly_salary, hours_per_week, hire_date, notes)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING; -- Optional: Handle potential duplicate runs
    """

    processed_count = 0
    inserted_count = 0
    skipped_loc_count = 0
    skipped_date_count = 0
    skipped_salary_count = 0

    print(f"Processing {len(employees_clean)} employee records...")
    for index, row in employees_clean.iterrows():
        processed_count += 1
        # Lookup location_id
        cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
        location_result = cur.fetchone()

        if location_result is None:
            skipped_loc_count += 1
            continue
        location_id = location_result[0]

        # Handle potential NaT/None in hire_date
        hire_date_to_insert = None
        if 'hire_date' in row and pd.notna(row['hire_date']):
            hire_date_to_insert = row['hire_date']
        else:
            skipped_date_count +=1

        # *** Get salary value from the 'salary' column in Excel/DataFrame ***
        salary_value = row.get('salary') # Use .get() for safety
        # *** Corrected typo in isinstance check ***
        if salary_value is None or not isinstance(salary_value, (int, float)):
             print(f"  - Warning: Missing or invalid 'salary' in source file for row {index}. Skipping.")
             skipped_salary_count += 1
             continue


        # Insert into employees table, mapping salary_value to yearly_salary column
        try:
            # *** Data tuple now passes salary_value for the yearly_salary DB column ***
            cur.execute(insert_sql, (
                location_id,
                row['first_name'],
                row['last_name'],
                row['department'],
                salary_value, # This value goes into the DB 'yearly_salary' column
                row['hours_per_week'],
                hire_date_to_insert,
                None  # notes column
            ))
            if cur.rowcount > 0:
                inserted_count += 1
        except psycopg2.Error as e:
             print(f"  - DB Error inserting row {index} ({row['first_name']} {row['last_name']}): {e}")
             print("  - Rolling back transaction and stopping.")
             conn.rollback()
             raise e

    print("\nFinished processing employees.")
    print(f"- Records inserted: {inserted_count}")
    print(f"- Records skipped (Location not found): {skipped_loc_count}")
    print(f"- Records skipped (Missing/Invalid Salary): {skipped_salary_count}")


    print("Committing transaction...")
    conn.commit()
    success = True

except KeyError as e:
    print(f"\nError: Missing expected column 'salary' in the Excel file: {e}")
    print("Please ensure the Excel file was generated with the script that includes 'salary'.")
    if conn: conn.rollback()
except psycopg2.OperationalError as e:
    print(f"\nDatabase Connection Error: {e}")
except psycopg2.Error as e:
    print(f"\nTransaction was rolled back due to a database error during processing.")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")
    if conn: conn.rollback()
finally:
    if cur: cur.close()
    if conn: conn.close()
    if success:
        print("Employees inserted successfully!") # Final success message
    else:
        print("Employee insertion failed or was rolled back.")

Successfully read 5 rows from sheet 'Employees' in /Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx
Converted 'hire_date' column, handling potential errors.
Connecting to database 'sql_final_project'...
Connection successful.
Processing 5 employee records...

Finished processing employees.
- Records inserted: 5
- Records skipped (Location not found): 0
- Records skipped (Missing/Invalid Salary): 0
Committing transaction...
Employees inserted successfully!


In [157]:
# 10) Staffing

staffing_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees_Staffing.xlsx')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in staffing_clean.iterrows():
    # Find employee_id using first_name + last_name
    cur.execute("""
        SELECT employee_id 
        FROM employees
        WHERE first_name = %s AND last_name = %s
        LIMIT 1
    """, (row['first_name'], row['last_name']))
    employee_result = cur.fetchone()

    if employee_result is None:
        print(f"Warning: Employee '{row['first_name']} {row['last_name']}' not found, skipping.")
        continue
    employee_id = employee_result[0]

    # Lookup location_id
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found for shift, skipping.")
        continue
    location_id = location_result[0]

    # Insert into staffing table
    cur.execute("""
        INSERT INTO staffing (employee_id, location_id, shift_date, start_time, end_time, status)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        employee_id,
        location_id,
        row['shift_date'],
        row['start_time'],
        row['end_time'],
        row['shift_status']
    ))

conn.commit()
cur.close()
conn.close()

print("Staffing inserted successfully!")

Staffing inserted successfully!


In [72]:
# 11) Customers

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

sales_customers_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')

for index, row in sales_customers_clean.iterrows():
    # Lookup location_id from location_name
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
    location_result = cur.fetchone()
    if location_result is None:
        print(f"Warning: Location '{row['location_name']}' not found, skipping.")
        continue
    location_id = location_result[0]

    # Clean missing customer info
    first_name = row['customer_first_name'] if pd.notna(row['customer_first_name']) else None
    last_name = row['customer_last_name'] if pd.notna(row['customer_last_name']) else None
    age = int(row['age']) if pd.notna(row['age']) else None
    gender = row['gender'] if pd.notna(row['gender']) else None
    email = row['email'] if pd.notna(row['email']) else None

    # Skip inserting if no first name or last name
    if first_name is None or last_name is None:
        print(f"Skipping row {index}: missing customer name.")
        continue

    # Loyalty handling
    if pd.isna(row['loyalty_member']):
        loyalty_member = False
    else:
        loyalty_member = True if str(row['loyalty_member']).lower() == 'yes' else False

    # Insert into customers
    cur.execute("""
        INSERT INTO customers (first_name, last_name, age, gender, email, location_id, loyalty_member)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        first_name,
        last_name,
        age,
        gender,
        email,
        location_id,
        loyalty_member
    ))

conn.commit()
cur.close()
conn.close()

print("Customers inserted successfully!")

Skipping row 4: missing customer name.
Skipping row 10: missing customer name.
Skipping row 14: missing customer name.
Skipping row 19: missing customer name.
Skipping row 23: missing customer name.
Skipping row 24: missing customer name.
Skipping row 31: missing customer name.
Skipping row 36: missing customer name.
Skipping row 37: missing customer name.
Skipping row 41: missing customer name.
Skipping row 50: missing customer name.
Skipping row 57: missing customer name.
Skipping row 59: missing customer name.
Skipping row 61: missing customer name.
Skipping row 65: missing customer name.
Skipping row 68: missing customer name.
Skipping row 69: missing customer name.
Skipping row 76: missing customer name.
Skipping row 79: missing customer name.
Skipping row 82: missing customer name.
Skipping row 83: missing customer name.
Skipping row 85: missing customer name.
Skipping row 87: missing customer name.
Skipping row 88: missing customer name.
Skipping row 89: missing customer name.
S

Not all buyers are customers (that gave their information and joined the loyalty program). Some those that do not have a name will be dropped.

In [74]:
# 12) Promotions

sales_customers = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')

# Only keep rows with a promotion name
sales_promotions = sales_customers[sales_customers['promotion_name'].notnull()]

# Aggregate
promotion_summary = sales_promotions.groupby('promotion_name').agg({
    'discount_applied': 'mean',
    'sale_date': ['min', 'max']
}).reset_index()

# Flatten multi-index columns
promotion_summary.columns = ['promotion_name', 'avg_discount', 'start_date', 'end_date']

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

cur.execute("SELECT location_id FROM locations;")
location_ids = [row[0] for row in cur.fetchall()]

# Step 6: Insert promotions
for idx, row in promotion_summary.iterrows():
    promotion_name = row['promotion_name']
    avg_discount = round(row['avg_discount'], 2)
    start_date = row['start_date']
    end_date = row['end_date']
    location_id = random.choice(location_ids) 
    
    cur.execute("""
        INSERT INTO promotions (promotion_name, start_date, end_date, location_id, discount_percentage, loyalty_only)
        VALUES (%s, %s, %s, %s, %s, %s);
    """, (
        promotion_name,
        start_date,
        end_date,
        location_id,
        avg_discount,
        False  # Default assumption that promotions are for everyone
    ))

# Step 7: Commit and close
conn.commit()
cur.close()
conn.close()

print("Promotions inserted successfully!")

Promotions inserted successfully!


For promotions, since there were already discount rates applied and the discount_percentage has a NOT NULL constraint, we computed the average for each type of deal.

In [76]:
# 13) Sales orders

sales_customers_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

for index, row in sales_customers_clean.iterrows():
    # Lookup location_id
    cur.execute("SELECT location_id FROM locations WHERE location_name = %s", (row['location_name'],))
    location = cur.fetchone()
    if not location:
        print(f"Location '{row['location_name']}' not found, skipping.")
        continue
    location_id = location[0]

    # SKIP if no customer names
    if pd.isna(row['customer_first_name']) or pd.isna(row['customer_last_name']):
        customer_id = None
    else:
        # Lookup customer_id normally
        cur.execute("""
            SELECT customer_id
            FROM customers
            WHERE first_name = %s AND last_name = %s
            LIMIT 1
        """, (row['customer_first_name'], row['customer_last_name']))
        customer = cur.fetchone()
        customer_id = customer[0] if customer else None

    # Insert into sales_orders
    cur.execute("""
        INSERT INTO sales_orders (order_date, location_id, customer_id)
        VALUES (%s, %s, %s)
    """, (
        row['sale_date'],
        location_id,
        customer_id
    ))

conn.commit()
cur.close()
conn.close()

print("Sales orders inserted successfully!")

Sales orders inserted successfully!


In [78]:
# 14) Sales order details
import pandas as pd
import psycopg2

sales_customers_clean = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Loop through the sales_customers_clean and insert into sales_orders_details
for index, row in sales_customers_clean.iterrows():
    # Find the sales_orders_id (matching by order_date and location)
    cur.execute("""
        SELECT sales_orders_id 
        FROM sales_orders 
        WHERE order_date = %s 
        AND location_id = (
            SELECT location_id FROM locations WHERE location_name = %s
        )
        LIMIT 1
    """, (row['sale_date'], row['location_name']))
    sales_orders_result = cur.fetchone()
    
    if sales_orders_result is None:
        print(f"Warning: Sale order not found for date {row['sale_date']} and location {row['location_name']}. Skipping.")
        continue
    sales_orders_id = sales_orders_result[0]

    # Find the product_id
    cur.execute("""
        SELECT product_id 
        FROM products 
        WHERE product_name = %s
        LIMIT 1
    """, (row['product_name'],))
    product_result = cur.fetchone()
    
    if product_result is None:
        print(f"Warning: Product '{row['product_name']}' not found. Skipping.")
        continue
    product_id = product_result[0]

    # Find the promotion_id (if promotion name is not null)
    promotion_id = None
    if pd.notna(row['promotion_name']):
        cur.execute("""
            SELECT promotion_id 
            FROM promotions 
            WHERE promotion_name = %s
            LIMIT 1
        """, (row['promotion_name'],))
        promo_result = cur.fetchone()
        if promo_result:
            promotion_id = promo_result[0]
        else:
            print(f"Warning: Promotion '{row['promotion_name']}' not found. Setting promotion_id = NULL.")

    # Step 4: Insert into sales_orders_details
    cur.execute("""
        INSERT INTO sales_orders_details (sales_orders_id, product_id, quantity, unit_price, discount, promotion_id)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        sales_orders_id,
        product_id,
        row['quantity'],
        row['unit_price'],
        row['discount_applied'],
        promotion_id
    ))

conn.commit()
cur.close()
conn.close()

print("Sales orders details inserted successfully!")

Sales orders details inserted successfully!


In [129]:
# 15 Expenses (Will take the purchase orders to get procurement, then load data from accounting.csv)
# Must locate the location_id since the file contains the name of the location where the expense ocurred.

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

accounting_expenses = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_accounting.xlsx')

# Lookup dictionaries
# Locations
cur.execute("SELECT location_id, location_name FROM locations;")
location_lookup = {name: loc_id for loc_id, name in cur.fetchall()}

# Suppliers
cur.execute("SELECT supplier_id, supplier_name FROM suppliers;")
supplier_lookup = {name: sid for sid, name in cur.fetchall()}

# Products
cur.execute("SELECT product_id, product_name FROM products;")
product_lookup = {name: pid for pid, name in cur.fetchall()}

# Insert Procurement Expenses (real quantities and unit_costs)
# Query to join purchase_orders + purchase_order_details
cur.execute("""
    SELECT po.purchase_order_id, po.order_date, po.location_id, po.supplier_id,
           pod.product_id, pod.quantity, pod.unit_cost
    FROM purchase_orders po
    JOIN purchase_order_details pod ON po.purchase_order_id = pod.purchase_order_id
    WHERE po.status = 'Received';
""")
procurement_rows = cur.fetchall()

for row in procurement_rows:
    purchase_order_id, order_date, location_id, supplier_id, product_id, quantity, unit_cost = row
    amount = round(quantity * unit_cost, 2)

    cur.execute("""
        INSERT INTO expenses (category, amount, expense_date, location_id, supplier_id, description)
        VALUES (%s, %s, %s, %s, %s, %s);
    """, (
        'Procurement',
        amount,
        order_date,
        location_id,
        supplier_id,
        f'Procurement of product_id {product_id} (qty: {quantity})'
    ))

print("Procurement expenses inserted.")

# Insert Operating Expenses (accounting CSV)

for index, row in accounting_expenses.iterrows():
    location_id = location_lookup.get(row['location_name'])
    if location_id is None:
        print(f"Warning: Location '{row['location_name']}' not found, skipping row.")
        continue

    cur.execute("""
        INSERT INTO expenses (category, amount, expense_date, location_id, supplier_id, description)
        VALUES (%s, %s, %s, %s, NULL, %s);
    """, (
        row['category'],
        row['amount'],
        row['expense_date'],
        location_id,
        row['description']
    ))

print("Operating expenses (ABC_accounting.xlsx) inserted.")

conn.commit()
cur.close()
conn.close()

print("Expenses inserted successfully!")

Procurement expenses inserted.
Operating expenses (ABC_accounting.xlsx) inserted.
Expenses inserted successfully!


In [123]:
# 16) Returns
# Since the client didn't document returns, we propose to make a different return table.
# This will help them track lost revenue and hidden costs.
# Fortunately, they documented some transactions in the sales and customers table where they added 'returned' in the notes.
# Therefore, we will only load the sales id, s.
# We will leave all the other as null values.

conn = psycopg2.connect(
    dbname='sql_final_project',
    user='postgres',
    password='123',
    host='localhost',
    port='5432'
)
cur = conn.cursor()

# Load sales and customers data
sales_customers = pd.read_excel('/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx')

# Step 1: Filter only returned sales
returned_sales = sales_customers[sales_customers['notes'] == 'Returned']

print(f"Found {len(returned_sales)} returned sales to insert into returns table.")

# Loop through and insert into returns table
for index, row in returned_sales.iterrows():
    # First lookup sales_orders_detail_id
    cur.execute("""
        SELECT sod.sales_orders_detail_id
        FROM sales_orders so
        JOIN sales_orders_details sod ON so.sales_orders_id = sod.sales_orders_id
        JOIN products p ON sod.product_id = p.product_id
        WHERE so.order_date = %s
          AND p.product_name = %s
          AND sod.quantity = %s
        LIMIT 1;
    """, (
        row['sale_date'],
        row['product_name'],
        row['quantity']
    ))
    result = cur.fetchone()

    if result is None:
        print(f"Warning: Could not find matching sale order for product {row['product_name']} on {row['sale_date']}. Skipping...")
        continue

    sales_orders_detail_id = result[0]

    # Insert into returns table
    cur.execute("""
        INSERT INTO returns (sales_orders_detail_id, quantity_returned, return_date, reason)
        VALUES (%s, %s, NULL, NULL)
    """, (
        sales_orders_detail_id,
        row['quantity']
    ))

conn.commit()
cur.close()
conn.close()

print("Returns inserted successfully!")

Found 51 returned sales to insert into returns table.
Returns inserted successfully!


This concludes the succesful data migration from the client to the newly formed schema.